# 03 2.5D Qwen2.5-VL SFT Train

Inputs:

- output dataset from notebook `01_build_25d_cache.ipynb`
- Qwen2.5-VL model available through Kaggle model/input or Hugging Face

Default is `SMOKE_MODE=True`. Use it first to verify data/model loading.
Then set `SMOKE_MODE=False` for a full QLoRA run.

Smoke gate: section validity >= 95%, output noise < 5%, val ROUGE-L >= 0.20.

In [ ]:
# If Kaggle image lacks packages, uncomment:
# !pip -q install -U "transformers>=4.51.0" accelerate peft bitsandbytes qwen-vl-utils pillow

from pathlib import Path
import json, warnings, os
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
from IPython.display import display
warnings.filterwarnings("ignore")

SMOKE_MODE = True
MODEL_ID_OR_PATH = os.environ.get("QWEN_MODEL_PATH", "Qwen/Qwen2.5-VL-3B-Instruct")
MAX_TRAIN_SAMPLES = 64 if SMOKE_MODE else None
MAX_VAL_SAMPLES = 32 if SMOKE_MODE else 128
OUTPUT_ROOT = Path("/kaggle/working/vimedpet_q1_outputs") if Path("/kaggle/working").exists() else Path("vimedpet_q1_outputs")
SFT_DIR = OUTPUT_ROOT / "03_qwen_sft"
SFT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def locate_cache():
    roots = list(Path("/kaggle/input").rglob("q1_cache_meta.json")) if Path("/kaggle/input").exists() else []
    roots += list(Path.cwd().rglob("q1_cache_meta.json"))
    if not roots: raise FileNotFoundError("q1_cache_meta.json not found. Add notebook 01 output.")
    meta_path = sorted(roots, key=lambda p: len(str(p)))[0]
    return meta_path.parent, json.loads(meta_path.read_text())
CACHE_DIR, meta = locate_cache()
clean = pd.read_csv(CACHE_DIR / "q1_clean_manifest_for_cache.csv")
cache_index = pd.read_csv(CACHE_DIR / "q1_cache_index.csv")
clean = clean.merge(cache_index[cache_index.cache_ok][["case_id", "row_index"]], on="case_id", how="inner")
arr = np.memmap(CACHE_DIR / meta["mmap_path"], dtype=np.uint8, mode="r", shape=(meta["n"], meta["channels"], meta["image_size"], meta["image_size"]))
train_df = clean[clean.clean_split == "train"].copy()
val_df = clean[clean.clean_split == "validation"].copy()
if MAX_TRAIN_SAMPLES: train_df = train_df.sample(min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=391)
if MAX_VAL_SAMPLES: val_df = val_df.sample(min(MAX_VAL_SAMPLES, len(val_df)), random_state=392)
print("train", train_df.shape, "val", val_df.shape, "cache", CACHE_DIR)

def row_to_rgb(row_index):
    x = arr[int(row_index)]
    # RGB mosaic: CT middle, PET coronal, PET axial
    rgb = np.stack([x[1], x[4], x[3]], axis=-1)
    return Image.fromarray(rgb, mode="RGB")

In [ ]:
from transformers import AutoProcessor, TrainingArguments, Trainer, BitsAndBytesConfig
try:
    from transformers import Qwen2_5_VLForConditionalGeneration as QwenModel
except Exception:
    from transformers import AutoModelForVision2Seq as QwenModel
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
processor = AutoProcessor.from_pretrained(MODEL_ID_OR_PATH, trust_remote_code=True)
model = QwenModel.from_pretrained(MODEL_ID_OR_PATH, quantization_config=bnb, device_map="auto", trust_remote_code=True)
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
PROMPT = "Generate a structured Vietnamese PET/CT report with two sections: Findings and Impression. Preserve clinically relevant organs, lesions, and SUV values when visible."

class PetReportDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        image = row_to_rgb(row.row_index)
        target = str(row.target_text or row.report_text_clean)
        messages = [
            {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": PROMPT}]},
            {"role": "assistant", "content": [{"type": "text", "text": target}]},
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        return {"text": text, "image": image}

def collate(batch):
    texts = [b["text"] for b in batch]
    images = [b["image"] for b in batch]
    enc = processor(text=texts, images=images, return_tensors="pt", padding=True)
    enc["labels"] = enc["input_ids"].clone()
    enc["labels"][enc["attention_mask"] == 0] = -100
    return enc

args = TrainingArguments(
    output_dir=str(SFT_DIR / "checkpoints"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1 if SMOKE_MODE else 2,
    max_steps=20 if SMOKE_MODE else -1,
    logging_steps=5,
    save_steps=20 if SMOKE_MODE else 100,
    eval_steps=20 if SMOKE_MODE else 100,
    eval_strategy="steps",
    save_total_limit=3,
    fp16=True,
    report_to="none",
    remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=PetReportDataset(train_df), eval_dataset=PetReportDataset(val_df), data_collator=collate)
trainer.train()
trainer.save_model(SFT_DIR / "adapter_final")
processor.save_pretrained(SFT_DIR / "processor")
print("Saved:", SFT_DIR)